# Integrate nSIDES with SIDER (PrimeKG)

**Purpose:** Explore Open nSIDES (onsides-v3.1.0) for integration with SIDER used in PrimeKG (drug–side effect). Run cell-by-cell, inspect outputs, then decide whether to use / integrate into PrimeKG.

**Main flow:**
1. Load SIDER (`sider.csv`: atc, UMLS_from_meddra, side_effect_name).
2. Load nSIDES: vocabs (MedDRA, RxNorm ingredient), high_confidence (ingredient_id, effect_meddra_id).
3. Map MedDRA effect → UMLS: match nSIDES effect names to SIDER to assign UMLS CUIs.
4. Build nSIDES table in SIDER-compatible format (prepare for RxNorm→ATC if mapping exists).
5. Compare overlap / mismatch with SIDER; export candidate table.

In [ ]:
from pathlib import Path
import os

def resolve_primekg_root() -> Path:
    if env := os.environ.get("PRIMEKG_ROOT"):
        root = Path(env).expanduser().resolve()
        if (root / "datasets" / "data" / "drugbank").is_dir():
            return root
        raise FileNotFoundError(f"PRIMEKG_ROOT invalid: {root}")
    for start in (Path.cwd().resolve(),):
        for parent in (start, *start.parents):
            if (parent / "datasets" / "data" / "drugbank").is_dir():
                return parent
            if parent.name == "sider_nsides" and (parent.parent.parent / "datasets" / "data" / "drugbank").is_dir():
                return parent.parent.parent
            if parent.name == "PrimeKG-Plus_release" and (parent.parent / "datasets" / "data" / "drugbank").is_dir():
                return parent.parent
    raise FileNotFoundError("Set PRIMEKG_ROOT to the PrimeKG repo root.")

PROJECT_ROOT = resolve_primekg_root()
DATA_DIR = PROJECT_ROOT / "datasets" / "data"
PREP_DIR = Path.cwd()  # run with CWD = source_prep/sider_nsides/
BASE = PREP_DIR
NSIDES_DIR = PREP_DIR / "inputs/nsides/csv"
SIDER_DIR = PREP_DIR / "inputs/sider"
OUTPUTS_DIR = PREP_DIR / "outputs"


In [1]:
import os
import pandas as pd
import numpy as np

print("PREP_DIR:", PREP_DIR)
print("nSIDES CSV:", NSIDES_DIR)
print("SIDER:", SIDER_DIR)
print("OUTPUTS:", OUTPUTS_DIR)
print("nSIDES exists:", NSIDES_DIR.is_dir())
print("SIDER exists:", SIDER_DIR.is_dir())


BASE: /Users/ljw303/YANG_DATA/PrimeKG/datasets/data/nSIDES
nSIDES CSV: /Users/ljw303/YANG_DATA/PrimeKG/datasets/data/nSIDES/onsides-v3.1.0/csv
SIDER: /Users/ljw303/YANG_DATA/PrimeKG/datasets/data/nSIDES/../sider
nSIDES exists: True
SIDER exists: True


## 1. Load SIDER (current PrimeKG format)

SIDER: `atc`, `UMLS_from_label`, `UMLS_from_meddra`, `side_effect_name`. PrimeKG merges with DrugBank via `atc` and maps effects to HPO via `UMLS_from_meddra`.

In [2]:
# Standard approach: use MedDRA
# In pharmacovigilance, MedDRA is the most widely used standard. Reasons:
# Regulatory requirement: FDA, EMA, PMDA, Health Canada, etc. require MedDRA for adverse-event / side-effect reporting.
# Shared vocabulary across regulators, companies, and studies enables global aggregation and comparison.
# Structured hierarchy (LLT → PT → HLT → HLGT → SOC), coded terms, controlled updates — not free-text label wording.
# Label text is raw input; MedDRA normalization is the industry standard. In SIDER, UMLS_from_meddra + side_effect_name are MedDRA-normalized; UMLS_from_label maps label text directly and is not MedDRA-standard.
# What is MedDRA?
# Full name: Medical Dictionary for Regulatory Activities.
# Purpose: standardized medical terminology for recording, querying, and analyzing data on drugs, vaccines, and devices (registration, safety surveillance, ADR reporting).
# Maintainer: ICH. Quality-controlled (e.g. ISO), updated twice per year.
# Scope: signs, symptoms, diseases, syndromes, diagnoses, device malfunctions, medication errors, procedures, indications, lab findings — including side effects / ADRs.
# Multilingual: English plus Chinese, French, German, Spanish, etc.
# MedDRA hierarchy (5 levels, broad → narrow)
# Level	Name	Brief meaning
# SOC	System Organ Class	Organ/system class (~27 SOCs).
# HLGT	High-Level Group Term	High-level group within a SOC.
# HLT	High-Level Term	Clinical concept level. ~1,600 HLTs.
# PT	Preferred Term	Standard coding/reporting level (e.g. “Abdominal pain”). ~16,000 PTs.
# LLT	Lowest Level Term	Synonyms / spelling variants of PTs. ~60,000 LLTs.
# Each PT is one medical concept; many LLTs map to one PT (e.g. “Belly ache”, “Stomach pain” → PT “Abdominal pain”).
# Multi-axial: a PT can belong to multiple SOCs (one primary, others secondary).
# In SIDER, filtering meddra_concept_type == "PT" uses MedDRA PT level; side_effect_name is the PT name, UMLS_from_meddra is its UMLS CUI.
# Quick comparison
# Label text only (free text / direct UMLS-from-label mapping):
# Useful raw source, but non-standard — varied wording, synonyms, multiple UMLS CUIs for one concept.
# MedDRA-normalized (PT) then use PT name or UMLS (UMLS_from_meddra):
# Industry/regulatory standard; suitable for nSIDES/onsides and other pharmacovigilance sources.
# Summary: prefer MedDRA-encoded side effects (PT level) — in SIDER use UMLS_from_meddra and side_effect_name. MedDRA is the global regulatory/pharmacovigilance dictionary.

In [3]:
# Why UMLS_from_meddra aligns with side_effect_name better than UMLS_from_label
# Per official SIDER README (meddra_all_se.tsv):
# Column 3 – UMLS_from_label: "UMLS concept id as it was found on the label"
# → UMLS CUI of label text (via NLP). One side effect may appear many ways, so one side_effect_name can map to many UMLS_from_label values.
# Column 5 – UMLS_from_meddra: "UMLS concept id for MedDRA term"
# → UMLS CUI of the MedDRA term (PT or LLT).
# Column 6 – side_effect_name is the MedDRA term name.
# → UMLS_from_meddra and side_effect_name are the same concept (CUI vs name); ~1:1 in the file.
# With meddra_concept_type == "PT", each row is one MedDRA PT with:
# side_effect_name = PT name (e.g. "Abdominal pain"),
# UMLS_from_meddra = UMLS CUI of that PT.
# UMLS lookup on UMLS_from_meddra returns the concept matching side_effect_name. UMLS_from_label is label-text CUI (LLT/synonym variants), so 1:1 mismatch with PT names is expected.
# Conclusion: for MedDRA-normalized UMLS (e.g. nSIDES), use UMLS_from_meddra + side_effect_name; use UMLS_from_label only for label-text analysis, not MedDRA PT.
df_sider = pd.read_csv(str(SIDER_DIR / "sider.csv"), low_memory=False)
print("Shape:", df_sider.shape)
print("Columns:", list(df_sider.columns))
print("\nSample:")
display(df_sider.head(5))
print("\nUnique ATC:", df_sider["atc"].nunique())
print("Unique (UMLS_from_meddra, side_effect_name):", df_sider.groupby(["UMLS_from_meddra", "side_effect_name"]).ngroups)
print("Unique side_effect_name:", df_sider["side_effect_name"].nunique())

Shape: (202736, 4)
Columns: ['atc', 'UMLS_from_label', 'UMLS_from_meddra', 'side_effect_name']

Sample:


,atc,UMLS_from_label,UMLS_from_meddra,side_effect_name
0,A16AA01,C0000729,C0000737,Abdominal pain
1,A16AA01,C0000737,C0687713,Gastrointestinal pain
2,A16AA01,C0000737,C0000737,Abdominal pain
3,A16AA01,C0002418,C0002418,Amblyopia
4,A16AA01,C0002871,C0002871,Anaemia



Unique ATC: 1560
Unique (UMLS_from_meddra, side_effect_name): 4073
Unique side_effect_name: 4073


In [4]:
df_sider = df_sider.drop_duplicates(subset=["atc", "UMLS_from_meddra", "side_effect_name"]).reset_index(drop=True)
len(df_sider)

174750

## 2. Load nSIDES: vocabs (MedDRA, RxNorm ingredient) and high_confidence

In [5]:
vocab_meddra = pd.read_csv(os.path.join(NSIDES_DIR, "vocab_meddra_adverse_effect.csv"))
#NLM definition of RxNorm: “A normalized naming system for generic and branded drugs; a list of medication names and dose forms.”
vocab_ingredient = pd.read_csv(os.path.join(NSIDES_DIR, "vocab_rxnorm_ingredient.csv"))
print("vocab_meddra:", vocab_meddra.shape)
print("vocab_ingredient:", vocab_ingredient.shape)
display(vocab_meddra.head(3))
display(vocab_ingredient.head(3))

vocab_meddra: (7177, 3)
vocab_ingredient: (2562, 3)


,meddra_id,meddra_name,meddra_term_type
0,10000021,21-hydroxylase deficiency,PT
1,10000045,Abdomen enlarged,LLT
2,10000050,Abdominal adhesions,PT


,rxnorm_id,rxnorm_name,rxnorm_term_type
0,1006297,Aspergillus niger var. niger allergenic extract,Ingredient
1,10814,liothyronine,Ingredient
2,10829,trimethoprim,Ingredient


In [6]:
df_hc = pd.read_csv(os.path.join(NSIDES_DIR, "high_confidence.csv"))
print("high_confidence shape:", df_hc.shape)
print("Columns:", list(df_hc.columns))
display(df_hc.head(10))

high_confidence shape: (663, 2)
Columns: ['ingredient_id', 'effect_meddra_id']


,ingredient_id,effect_meddra_id
0,68149,10037844
1,6851,10024382
2,10395,10019211
3,1100072,10017076
4,5487,10008479
5,35636,10021425
6,35636,10016173
7,342369,10003549
8,25255,10067484
9,73494,10020751


## 3. Map MedDRA (nSIDES) → UMLS using SIDER
# Unique (side_effect_name, UMLS_from_meddra) from SIDER; 
# match nSIDES meddra_name (lowercase, strip) to assign UMLS to effects.

In [7]:
# Lookup: side_effect_name (normalized) -> UMLS_from_meddra
sider_effect_lookup = df_sider.drop_duplicates(subset=["side_effect_name", "UMLS_from_meddra"])[["side_effect_name", "UMLS_from_meddra"]]
sider_effect_lookup["_key"] = sider_effect_lookup["side_effect_name"].str.strip().str.lower()
# sider_effect_lookup.head(5)
sider_lookup = sider_effect_lookup.drop_duplicates(subset=["_key"]).set_index("_key")["UMLS_from_meddra"]
sider_lookup.head(5)

_key
abdominal pain           C0000737
gastrointestinal pain    C0687713
amblyopia                C0002418
anaemia                  C0002871
decreased appetite       C0232462
Name: UMLS_from_meddra, dtype: object

In [8]:
# Map nSIDES meddra_name -> UMLS
vocab_meddra["_key"] = vocab_meddra["meddra_name"].astype(str).str.strip().str.lower()
vocab_meddra.head(5)
vocab_meddra["umls_cui"] = vocab_meddra["_key"].map(sider_lookup)
vocab_meddra.head(5)
matched = vocab_meddra["umls_cui"].notna().sum()
matched
print("MedDRA terms in nSIDES vocab:", len(vocab_meddra))
print("Matched to UMLS via SIDER name:", matched)
print("Unmatched sample:")
# display(vocab_meddra.head(50))
display(vocab_meddra[vocab_meddra["umls_cui"].isna()].head(5))

MedDRA terms in nSIDES vocab: 7177
Matched to UMLS via SIDER name: 2771
Unmatched sample:


,meddra_id,meddra_name,meddra_term_type,_key,umls_cui
0,10000021,21-hydroxylase deficiency,PT,21-hydroxylase deficiency,NaN
1,10000045,Abdomen enlarged,LLT,abdomen enlarged,NaN
3,10000054,Abdominal aortic aneurysm,LLT,abdominal aortic aneurysm,NaN
4,10000055,Abdominal colic,LLT,abdominal colic,NaN
5,10000056,Abdominal cramp,LLT,abdominal cramp,NaN


In [9]:
# Effect table: meddra_id -> meddra_name, umls_cui
effect_map = vocab_meddra[["meddra_id", "meddra_name", "umls_cui"]].drop_duplicates()
effect_map = effect_map[effect_map["umls_cui"].notna()].copy()
print("Effect map (meddra_id -> UMLS) rows:", len(effect_map))
display(effect_map.head(5))
#Check from UMLS: Abdominal adhesions (C0549357) 

Effect map (meddra_id -> UMLS) rows: 2771


,meddra_id,meddra_name,umls_cui
2,10000050,Abdominal adhesions,C0549357
7,10000059,Abdominal discomfort,C0232487
8,10000060,Abdominal distension,C0000731
14,10000081,Abdominal pain,C0000737
15,10000084,Abdominal pain lower,C0232495


## 4. nSIDES table in SIDER-compatible format
<!-- High-confidence: (ingredient_id, effect_meddra_id). 
Add ingredient_name, effect_name, umls_cui. 
Leave `atc` empty (RxNorm→ATC mapping needed later). -->

In [10]:
# Join high_confidence with vocabs (align dtypes for merge)
df_hc = df_hc.astype({"ingredient_id": str})
vocab_ingredient["rxnorm_id"] = vocab_ingredient["rxnorm_id"].astype(str)

nsides_sider_like = (
    df_hc
    .merge(vocab_ingredient.rename(columns={"rxnorm_id": "ingredient_id",
                                            "rxnorm_name": "ingredient_name"}),
           on="ingredient_id", how="left")
    .merge(effect_map.rename(columns={"meddra_id": "effect_meddra_id",
                                      "meddra_name": "effect_name"}),
           on="effect_meddra_id", how="inner")  # keep only UMLS-mapped effects
)

nsides_sider_like["atc"] = np.nan
nsides_sider_like["UMLS_from_meddra"] = nsides_sider_like["umls_cui"]
nsides_sider_like["side_effect_name"] = nsides_sider_like["effect_name"]

print("Rows (high_confidence with UMLS-mapped effect):", len(nsides_sider_like))
display(nsides_sider_like[["ingredient_id", "ingredient_name",
                           "effect_meddra_id", "effect_name",
                           "UMLS_from_meddra", "side_effect_name",
                           "atc"]].head(10))

Rows (high_confidence with UMLS-mapped effect): 550


,ingredient_id,ingredient_name,effect_meddra_id,effect_name,UMLS_from_meddra,side_effect_name,atc
0,68149,mycophenolate mofetil,10037844,Rash,C0015230,Rash,NaN
1,6851,methotrexate,10024382,Leukoencephalopathy,C0270612,Leukoencephalopathy,NaN
2,10395,tetracycline,10019211,Headache,C0018681,Headache,NaN
3,1100072,abiraterone,10017076,Fracture,C0016658,Fracture,NaN
4,5487,hydrochlorothiazide,10008479,Chest pain,C0008031,Chest pain,NaN
5,35636,risperidone,10021425,Immune system disorder,C0021053,Immune system disorder,NaN
6,35636,risperidone,10016173,Fall,C0085639,Fall,NaN
7,342369,lenalidomide,10003549,Asthenia,C0004093,Asthenia,NaN
8,73494,telmisartan,10020751,Hypersensitivity,C0020517,Hypersensitivity,NaN
9,190521,abacavir,10003239,Arthralgia,C0003862,Arthralgia,NaN


## 4b. RxNorm ingredient → ATC (from DrugBank XML)

DrugBank XML has `<external-identifier>` with `<resource>RxCUI</resource>` and `<atc-code code="...">`. Parse to (drugbank_id, atc_code, rxcui), build **rxnorm_id → atc**, merge into `nsides_sider_like` to fill `atc`.

In [11]:
# DrugBank XML path (same data as sider: datasets/data/drugbank)
DRUGBANK_DIR = DATA_DIR / "drugbank"
# Prefer newer full build (Nurset); fallback database.xml in drugbank folder
_dbx = DRUGBANK_DIR / "Nurset_data_drugbank" / "2025_01_ver_full_database.xml"
if not Path(_dbx).is_file():
    _dbx = DRUGBANK_DIR / "full database.xml"
DRUGBANK_XML = _dbx
print("DrugBank XML:", DRUGBANK_XML)
print("Exists:", Path(DRUGBANK_XML).is_file())

DrugBank XML: /Users/ljw303/YANG_DATA/PrimeKG/datasets/data/nSIDES/../drugbank/Nurset_data_drugbank/2025_01_ver_full_database.xml
Exists: True


In [12]:
# Parse DrugBank XML: (drugbank_id, atc_code, rxcui) via iterparse to avoid loading entire file
import xml.etree.ElementTree as ET

NS = {"db": "http://www.drugbank.ca"}

def _tag(name):
    return f"{{http://www.drugbank.ca}}{name}"

rows = []
if os.path.isfile(DRUGBANK_XML):
    for event, elem in ET.iterparse(DRUGBANK_XML, events=("end",)):
        if elem.tag != _tag("drug"):
            continue
        primary = elem.find(f".//{_tag('drugbank-id')}[@primary='true']")
        if primary is None:
            primary = elem.find(f".//{_tag('drugbank-id')}")
        drugbank_id = primary.text.strip() if primary is not None and primary.text else None
        if not drugbank_id:
            elem.clear()
            continue
        atc_codes = []
        for atc in elem.findall(f".//{_tag('atc-code')}"):
            code = atc.get("code")
            if code:
                atc_codes.append(code.strip())
        rxcui = None
        for ext in elem.findall(f".//{_tag('external-identifier')}"):
            res = ext.find(_tag("resource"))
            ident = ext.find(_tag("identifier"))
            if res is not None and ident is not None and res.text and "RxCUI" in res.text:
                rxcui = ident.text.strip() if ident.text else None
                break
        if rxcui and atc_codes:
            for atc in atc_codes:
                rows.append({"drugbank_id": drugbank_id, "atc_code": atc, "rxnorm_id": rxcui})
        elem.clear()
else:
    print("DrugBank XML not found; rxnorm_to_atc will be empty.")

df_drugbank_rxnorm_atc = pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame(columns=["drugbank_id", "atc_code", "rxnorm_id"])
rxnorm_to_atc = df_drugbank_rxnorm_atc[["rxnorm_id", "atc_code"]].drop_duplicates() if len(df_drugbank_rxnorm_atc) else pd.DataFrame(columns=["rxnorm_id", "atc_code"])
print("DrugBank (drugbank_id, atc_code, rxnorm_id) rows:", len(df_drugbank_rxnorm_atc))
print("Unique (rxnorm_id, atc_code) rows:", len(rxnorm_to_atc))
display(rxnorm_to_atc.head(10))

DrugBank (drugbank_id, atc_code, rxnorm_id) rows: 4876
Unique (rxnorm_id, atc_code) rows: 4876


,rxnorm_id,atc_code
0,237057,B01AE02
1,318341,L01FE01
2,337623,R05CB13
3,214470,L01XX29
4,214555,L04AB01
5,60819,B01AE06
6,42375,L02AE51
7,42375,L02AE02
8,120608,L03AB11
9,120608,L03AB61


In [13]:
# Merge RxNorm → ATC into nsides_sider_like (one ingredient may have many ATCs; take first per rxnorm_id)
rxnorm_to_atc["rxnorm_id"] = rxnorm_to_atc["rxnorm_id"].astype(str)
first_atc = rxnorm_to_atc.drop_duplicates(subset=["rxnorm_id"], keep="first").set_index("rxnorm_id")["atc_code"]
nsides_sider_like["atc"] = nsides_sider_like["ingredient_id"].map(first_atc)

mapped = nsides_sider_like["atc"].notna().sum()
print("Rows with ATC filled (RxNorm mapped via DrugBank):", mapped, "/", len(nsides_sider_like))
print("Unique ingredients in nSIDES table:", nsides_sider_like["ingredient_id"].nunique())
print("Unique ingredients with ATC:", nsides_sider_like.loc[nsides_sider_like["atc"].notna(), "ingredient_id"].nunique())
print("Len nsides_sider_like: ", len(nsides_sider_like))
display(nsides_sider_like[["ingredient_id", "ingredient_name", "atc", "UMLS_from_meddra", "side_effect_name"]].head(10))

Rows with ATC filled (RxNorm mapped via DrugBank): 527 / 550
Unique ingredients in nSIDES table: 62
Unique ingredients with ATC: 60
Len nsides_sider_like:  550


,ingredient_id,ingredient_name,atc,UMLS_from_meddra,side_effect_name
0,68149,mycophenolate mofetil,NaN,C0015230,Rash
1,6851,methotrexate,L04AX03,C0270612,Leukoencephalopathy
2,10395,tetracycline,S02AA08,C0018681,Headache
3,1100072,abiraterone,L01XK52,C0016658,Fracture
4,5487,hydrochlorothiazide,C09BX03,C0008031,Chest pain
5,35636,risperidone,N05AX08,C0021053,Immune system disorder
6,35636,risperidone,N05AX08,C0085639,Fall
7,342369,lenalidomide,L04AX04,C0004093,Asthenia
8,73494,telmisartan,C09DB04,C0020517,Hypersensitivity
9,190521,abacavir,J05AR13,C0003862,Arthralgia


In [14]:
out_cols = ["ingredient_id", "ingredient_name", "atc",
            "UMLS_from_meddra", "side_effect_name"]

out_path = PREP_DIR / "outputs/nsides_high_confidence_sider_like.csv"

nsides_sider_like[out_cols].to_csv(out_path, index=False)

print("Saved:", out_path)
print("Rows:", len(nsides_sider_like))

Saved: /Users/ljw303/YANG_DATA/PrimeKG/datasets/data/nSIDES/nsides_high_confidence_sider_like.csv
Rows: 550


In [15]:
nsides_sider_like = nsides_sider_like[["atc", "UMLS_from_meddra", "side_effect_name"]]
nsides_sider_like = nsides_sider_like.dropna()
print("Len nsides_sider_like: ", len(nsides_sider_like))

Len nsides_sider_like:  527


In [16]:
nsides_sider_like["UMLS_from_label"] = "-"
nsides_sider_like = nsides_sider_like[df_sider.columns]

In [17]:
nsides_sider_like.head(3)

,atc,UMLS_from_label,UMLS_from_meddra,side_effect_name
1,L04AX03,-,C0270612,Leukoencephalopathy
2,S02AA08,-,C0018681,Headache
3,L01XK52,-,C0016658,Fracture


In [18]:
print(len(df_sider))
df_sider.head(5)

174750


,atc,UMLS_from_label,UMLS_from_meddra,side_effect_name
0,A16AA01,C0000729,C0000737,Abdominal pain
1,A16AA01,C0000737,C0687713,Gastrointestinal pain
2,A16AA01,C0002418,C0002418,Amblyopia
3,A16AA01,C0002871,C0002871,Anaemia
4,A16AA01,C0003123,C0232462,Decreased appetite


In [19]:
new_df_sider = pd.concat([df_sider, nsides_sider_like], axis=0)
print(len(new_df_sider))
new_df_sider

175277


,atc,UMLS_from_label,UMLS_from_meddra,side_effect_name
0,A16AA01,C0000729,C0000737,Abdominal pain
1,A16AA01,C0000737,C0687713,Gastrointestinal pain
2,A16AA01,C0002418,C0002418,Amblyopia
3,A16AA01,C0002871,C0002871,Anaemia
4,A16AA01,C0003123,C0232462,Decreased appetite
...,...,...,...,...
545,L01XX35,-,C0013395,Dyspepsia
546,L04AX04,-,C0007859,Neck pain
547,L01CD01,-,C0027765,Nervous system disorder
548,L01XG01,-,C0018681,Headache


In [20]:
df_sider = df_sider.drop_duplicates(subset=["atc", "UMLS_from_meddra", "side_effect_name"]).reset_index(drop=True)
len(df_sider)

174750

In [21]:
new_df_sider = new_df_sider.drop_duplicates(subset=["atc", "UMLS_from_meddra", "side_effect_name"]).reset_index(drop=True)
len(new_df_sider)

175005

In [22]:
new_df_sider

,atc,UMLS_from_label,UMLS_from_meddra,side_effect_name
0,A16AA01,C0000729,C0000737,Abdominal pain
1,A16AA01,C0000737,C0687713,Gastrointestinal pain
2,A16AA01,C0002418,C0002418,Amblyopia
3,A16AA01,C0002871,C0002871,Anaemia
4,A16AA01,C0003123,C0232462,Decreased appetite
...,...,...,...,...
175000,L01XG01,-,C0027796,Neuralgia
175001,L01XG01,-,C0000737,Abdominal pain
175002,R03AK15,-,C0017601,Glaucoma
175003,L04AX04,-,C0032343,Poisoning


In [23]:
out_path = OUTPUTS_DIR / "sider_with_nsides.csv"
new_df_sider.to_csv(out_path, index=False)